# ISIC 2017 — Full 600-Image Evaluation Pipeline (end-to-end)

A single top-to-bottom Colab run that produces **all** Chapter 4 artifacts at
600-image scale:

| # | Stage | Output CSV | Est. runtime (T4) |
|---|---|---|---|
| 6 | Main scores: 600 img × 2 models × 7 FAE × 12 metrics | `full_run_7fae_12metrics_600.csv` | **~3.5–4.5 h** |
| 7 | Coverage / NaN report (incl. MPR) | *(printed + `full_run_coverage_600.csv`)* | < 1 min |
| 8 | Redundancy matrices per model | `redundancy_matrix_{resnet18,squeezenet}.csv` | < 1 min |
| 9 | Meta-evaluation reliability (NR + AR) | `meta_evaluation_reliability_FULL.csv` | **~4–5 h** |
| 11 | Four-scheme ranking comparison | `ranking_comparison_FULL.csv` | < 1 min |
| 12 | Ensemble (NormEnsembleXAI) scores | `full_run_ensemble_7fae_12metrics_600.csv` | **~1–1.5 h** |
| 13 | Ensemble four-scheme ranking | `ensemble_ranking_FULL.csv` | < 1 min |
| 14 | Wilcoxon mqdiscount vs single_fc (sanity) | *(printed)* | < 1 s |
| 15 | Verify all artifacts on Drive | — | < 1 min |
| | **TOTAL** | | **~9–12 h** |

> **Wall-clock**: budget **9–12 hours** on a T4. **Colab Pro (12 h sessions)
> is strongly recommended.** Cells 6, 9 and 12 all **checkpoint incrementally
> and resume** after a disconnect — just re-run the cell. Nothing is lost.

### Durable paths & the protected scores file (read this!)
After a near-miss where a coverage output collided with the scores file's Drive
path and **overwrote the 600-image scores CSV**, all path handling is hardened:

* **`SCORES_FINAL`** (`MyDrive/thesis/results/full_run_7fae_12metrics_600.csv`)
  is the **protected scores file**. It is written by **exactly one** cell — the
  main-scores cell (6) — at successful completion. Nothing else may touch it.
* **`DRIVE_RESULTS`** (`MyDrive/thesis/results`) holds **every** artifact, each
  with a **distinct basename** (coverage, redundancy, meta-eval, ranking,
  ensemble, ensemble-ranking). All artifacts are therefore **durable on Drive**
  and survive a runtime reset — not just the final copy.
* Every downstream cell **reads scores from `SCORES_FINAL`** (with a graceful
  fallback to the recovered `SCORES_full_run_600_GOOD.csv`, then the local copy)
  and **writes its outputs via `_safe_out(...)`** — a guard that *asserts* the
  target is not `SCORES_FINAL`, making a repeat of the overwrite **impossible**.

> The config cell (cell 2) defines `DRIVE_RESULTS`, `SCORES_FINAL`,
> `SCORES_RECOVERED`, `SCORES_LOCAL`, `_safe_out()` and `_resolve_scores()`.

### Durable checkpoints — survive a runtime recycle
Checkpoints are **Drive-durable**. Each long cell (6, 9, 12) writes its resume
CSV to the fast local `/content` disk for per-row appends, then snapshots the
*whole file* to Drive every `SYNC_EVERY` items (and once more at the end). The
local `/content` disk is wiped when Colab recycles the runtime, but the Drive
snapshot is not — so progress is no longer lost on disconnect.

**After a runtime reset**, you do **not** restart from zero:
1. Re-run cells **1–5** (mount Drive + define durable paths, clone repo, install
   deps, copy weights, copy data).
2. To **continue an interrupted long cell** (6, 9 or 12), re-run that cell — it
   **restores its checkpoint from Drive** and resumes.
3. To run a **downstream cell** (7, 8, 11, 13, 14, 15) directly, just run it —
   it **reads the scores from `SCORES_FINAL` on Drive** and writes its output
   back to Drive. No need to re-run the expensive scores cell.

> **Note — `attributions_cache/` is local-only** and lives on `/content`; it is
> **not** synced to Drive. After a reset it simply recomputes (cheap relative to
> the metric scores, which the Drive-durable CSVs protect).

### Runtime model (from `results/profile_t4.csv`)
The profiled T4 per-image cost is dominated by **occlusion** attributions and
two metrics that recompute attributions under perturbation:

* `occlusion` + `model_parameter_randomisation`: **~102 s / image** (single cell!)
* `occlusion` + `max/avg_sensitivity`: ~24 s / image each
* everything else (gradient methods): < 4 s / image / metric

Across 7 FAE × 2 models the heavy tail is occlusion's MPR/sensitivity; cached
attributions remove the attribution cost on re-runs but **metric** cost remains.

### Explicit decision — `non_sensitivity`
`non_sensitivity` is **DISABLED** for the full run (`INCLUDE_NON_SENSITIVITY =
False` in cell 6). Rationale: even on the 56×56 down-sampled path it costs
~34 s/image and adds **8,400** all-NaN-equivalent rows for 600 images, with no
payoff (it is excluded from $M^*$ and from every Chapter 4 table). The flag is
parameterised — set it to `True` only for a deliberate axiomatic sub-study, and
expect **+5–6 h**.

### Known issue — `model_parameter_randomisation` NaNs
In the 12-image pilot MPR produced 71/168 NaN (an intermittent Quantus failure,
likely SqueezeNet layer-ordering). Cell 6 now **captures every per-(model,
image, FAE, metric) failure** into an `error` column instead of silently
emitting NaN, and cell 7 prints a coverage report so the MPR coverage is a
recorded, inspectable number rather than a mystery.

### Prerequisites
* Runtime → Change runtime type → **T4 GPU**.
* Google Drive contains:
  * `thesis/weights/resnet18_isic2017.pth`, `thesis/weights/squeezenet_isic2017.pth`
  * `thesis/data/images/test/` and `thesis/data/masks/test/` (600 images + masks,
    saved by `colab_vertical_slice.ipynb`; falls back to ISIC download).


## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_THESIS = '/content/drive/MyDrive/thesis'

import os
os.makedirs(f'{DRIVE_THESIS}/results', exist_ok=True)
print(f'Drive mounted. Thesis folder: {DRIVE_THESIS}')

# ----- Durable paths config (hardened after the scores-overwrite incident) ---
# All artifacts now live on Drive (durable across runtime resets), each with a
# DISTINCT basename. The scores file is *protected*: only the scores cell (12)
# may write SCORES_FINAL; _safe_out() refuses any other write to that path.
DRIVE_RESULTS = f'{DRIVE_THESIS}/results'
os.makedirs(DRIVE_RESULTS, exist_ok=True)

# The protected scores file — ONLY the scores cell writes this.
SCORES_FINAL = f'{DRIVE_RESULTS}/full_run_7fae_12metrics_600.csv'

# Recovered-scores fallback (the user recovered the clobbered scores here and
# will also copy it to SCORES_FINAL). Downstream cells fall back to this.
SCORES_RECOVERED = f'{DRIVE_RESULTS}/SCORES_full_run_600_GOOD.csv'
# Last-resort local fallback (wiped on runtime reset; used only if neither
# Drive copy exists).
SCORES_LOCAL = 'results/full_run_7fae_12metrics_600.csv'

def _safe_out(path):
    """Guard against clobbering the protected scores file. Returns *path*
    unchanged unless it resolves to SCORES_FINAL, in which case it refuses."""
    assert os.path.abspath(path) != os.path.abspath(SCORES_FINAL), \
        f'REFUSING to overwrite the protected scores file: {path}'
    return path

def _resolve_scores():
    """Pick the durable scores source for downstream cells, preferring the
    protected file, then the recovered file, then the local copy."""
    for _src in (SCORES_FINAL, SCORES_RECOVERED, SCORES_LOCAL):
        if os.path.exists(_src):
            print(f'Reading scores from: {_src}')
            return _src
    raise FileNotFoundError(
        f'No scores CSV found. Looked for: {SCORES_FINAL}, '
        f'{SCORES_RECOVERED}, {SCORES_LOCAL}')

print(f'Durable results dir: {DRIVE_RESULTS}')
print(f'Protected scores file: {SCORES_FINAL}')

## 2. Clone Repository

In [ ]:
import os

REPO_DIR = '/content/fae-metrics-master-thesis'

# The GitHub repo `dawkopagh/fae-metrics-master-thesis` is the THESIS parent;
# the runnable code (src/, weights/, data/, results/) lives in its
# fae-metrics-master-thesis/ SUBDIRECTORY. So we clone the parent, then cd into
# the code subdir (double-nested path). cd-ing one level too shallow -> "No
# module named 'src'"; one too deep -> also fails.
if not os.path.exists(REPO1_DIR):
    !git clone https://github.com/dawkopagh/fae-metrics-master-thesis.git {REPO_DIR}

CODE_DIR = os.path.join(REPO_DIR, 'fae-metrics-master-thesis')
%cd {CODE_DIR}
!git pull

print(f'Working directory: {os.getcwd()}')
assert os.path.isdir('src'), f"src/ not found in {os.getcwd()} - clone/layout problem"
print('src/ found OK.  HEAD commit:', end=' ')
!git rev-parse HEAD


## 3. Install Dependencies

Pins `quantus==0.6.0` explicitly (same as the other notebooks). No extra deps.

In [ ]:
!pip install -q -r requirements.txt
!pip install -q quantus==0.6.0

import captum, quantus, torch, scipy
print(f'captum  {captum.__version__}')
print(f'quantus {quantus.__version__}')
print(f'torch   {torch.__version__}')
print(f'scipy   {scipy.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## 4. Copy Weights from Drive

SHA-256 values are from `weights/README.md` (trained 2026-04-22, full ISIC 2017).

In [ ]:
DRIVE_THESIS   = '/content/drive/MyDrive/thesis'
RESNET_SHA     = '593dcb844b8359550e3d84667475bbd845f45a6cb388356480c0a55cb5173430'
SQUEEZENET_SHA = '8bbb43bbba4ee81e58e354295420bea33e55cfa2be11c31d82dce272d03b092d'

import os
os.makedirs('weights', exist_ok=True)

!cp {DRIVE_THESIS}/weights/resnet18_isic2017.pth   weights/resnet18_isic2017.pth
!cp {DRIVE_THESIS}/weights/squeezenet_isic2017.pth weights/squeezenet_isic2017.pth

print('SHA-256 checksums (actual):')
!sha256sum weights/resnet18_isic2017.pth weights/squeezenet_isic2017.pth
print(f'\nExpected resnet18:   {RESNET_SHA}')
print(f'Expected squeezenet: {SQUEEZENET_SHA}')

## 5. Copy Full Test Data from Drive

Copies 600 test images + 600 segmentation masks (~800 MB) from Drive.
These were saved by `colab_vertical_slice.ipynb`.
Falls back to a full ISIC 2017 download (~6 GB, ~20 min) if not found.

In [ ]:
DRIVE_THESIS = '/content/drive/MyDrive/thesis'

import os, sys
sys.path.insert(0, '.')

_drive_has_images = os.path.exists(f'{DRIVE_THESIS}/data/images/test')
_drive_has_masks  = os.path.exists(f'{DRIVE_THESIS}/data/masks/test')

if _drive_has_images and _drive_has_masks:
    # Fast path: restore the cached test split from Drive (no re-download).
    for split_dir in ['data/images/test', 'data/masks/test']:
        src = f'{DRIVE_THESIS}/{split_dir}'
        os.makedirs(split_dir, exist_ok=True)
        !cp -r {src}/* {split_dir}/
        n = sum(len(files) for _, _, files in os.walk(split_dir))
        print(f'{split_dir}: {n} files restored from Drive (no download).')
else:
    # First run only: download just the 600-image TEST split (not train/val,
    # which would add ~5 GB), then cache it to Drive so every future runtime
    # skips the download. (Cell 30 also caches at the end, but a disconnect in
    # the long cells never reaches it — so we cache here, right after download.)
    print('Test data not on Drive — downloading ISIC 2017 TEST split only (~1.5 GB).')
    from src.data.download_isic import download_isic2017
    download_isic2017(dest_dir='data', skip_existing=True, splits=['test'])

    print('Caching test split to Drive for future runs...')
    for split_dir in ['data/images/test', 'data/masks/test']:
        _dest = f'{DRIVE_THESIS}/{split_dir}'
        if os.path.exists(split_dir):
            os.makedirs(_dest, exist_ok=True)
            !cp -r {split_dir}/* {_dest}/
            print(f'  cached {split_dir} -> Drive')

_test_classes = ['melanoma', 'nevus', 'seborrheic_keratosis']
n_images = sum(
    len([f for f in os.listdir(f'data/images/test/{cls}') if not f.startswith('.')])
    for cls in _test_classes
    if os.path.exists(f'data/images/test/{cls}')
)
print(f'\nTest images found: {n_images}')
assert n_images == 600, f'Expected 600 test images, got {n_images}'
print('Data scaffold ready.')


## 6. Main Scores — 600 Images, robust + resumable (~3.5–4.5 h)

This is the heart of the run: **2 models × 7 FAE × 600 images × 12 metrics**.
Instead of `experiments/vertical_slice.py` (which writes once at the very end),
this cell drives `src.pipeline` primitives directly so it can:

1. **Checkpoint after every (image, model, FAE)** — rows are appended to
   `full_run_7fae_12metrics_600.csv` immediately. A disconnect loses at most
   one FAE method's worth of work.
2. **Resume** — already-written `(model, image_id, fae_method)` triples are
   skipped on re-run.
3. **Capture failures per (model, image, FAE, metric)** into an `error` column
   so the MPR NaN issue is *recorded*, not silently dropped (cell 7 reports it).
4. **Cache attributions** under `attributions_cache/` (gradient methods are then
   near-free on re-runs).

`INCLUDE_NON_SENSITIVITY = False` (see intro for the explicit rationale).

In [ ]:
import os, sys, time, datetime, traceback, shutil
import numpy as np, pandas as pd, torch
sys.path.insert(0, '.')
os.makedirs('results', exist_ok=True)

# ----- Configuration ------------------------------------------------------
SCORES_CSV = 'results/full_run_7fae_12metrics_600.csv'
INCLUDE_NON_SENSITIVITY = False   # explicit decision — see intro markdown
INCLUDE_MPR = False               # MPRT broken in Quantus 0.6.0 on ISIC 2017 (AssertionError every sample); RandomLogit still covers Randomization
SEED = 42

# ----- Drive-durable checkpoint -------------------------------------------
# Local append is fast; Drive FUSE is terrible at frequent small appends, so we
# keep the local append and push a *whole-file snapshot* to Drive every
# SYNC_EVERY images. The local /content disk is wiped on a runtime recycle, but
# the Drive snapshot survives, so a fresh runtime restores it and resumes.
DRIVE_THESIS = '/content/drive/MyDrive/thesis'
DRIVE_CKPT = f'{DRIVE_THESIS}/results/checkpoints/{os.path.basename(SCORES_CSV)}'
SYNC_EVERY = 25   # snapshot local CSV -> Drive every N images (loss window <= N imgs)
LOG_EVERY = 1     # progress line every N images
SLOW_TRIPLE_S = 10  # also log any single (model,FAE) triple slower than this (catches occlusion)

def _sync_to_drive(local, drive):
    """Push a full-file snapshot of *local* to *drive*; never crash the run."""
    try:
        os.makedirs(os.path.dirname(drive), exist_ok=True)
        shutil.copy2(local, drive)
        n_rows = sum(1 for _ in open(local)) - 1  # minus header
        print(f'  synced {max(n_rows, 0)} rows to Drive ({drive})')
    except Exception as exc:
        print(f'  WARN: Drive sync failed (continuing): {type(exc).__name__}: {exc}')

from src.attributions.generate import (
    FAE_METHODS, compute_or_load, compute_gradcam, compute_integrated_gradients,
    compute_occlusion,
)
from src.attributions.cache import AttributionCache
from src.data.isic_dataset import ISIC2017Dataset
from src.metrics.quantus_wrapper import compute_all_metrics
from src.models.classifiers import (
    get_gradcam_target_layer, load_resnet18, load_squeezenet,
)
from src.pipeline import _make_explain_func

import random
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
print(f'Device: {device}')

cache = AttributionCache(root='attributions_cache')
models_dict = {
    'resnet18':   (load_resnet18('weights/resnet18_isic2017.pth', device=device), 'resnet18'),
    'squeezenet': (load_squeezenet('weights/squeezenet_isic2017.pth', device=device), 'squeezenet'),
}
dataset = ISIC2017Dataset(root_dir='data', split='test', image_size=224, return_mask=True)
N_IMAGES = len(dataset)
fae_names = list(FAE_METHODS.keys())
print(f'{N_IMAGES} images × {len(models_dict)} models × {len(fae_names)} FAE')

# ----- Restore-on-start: pull prior progress back from Drive --------------
# If the local CSV is gone (fresh runtime) but a Drive snapshot exists, copy it
# back BEFORE building the resume set so we pick up where we left off.
if not os.path.exists(SCORES_CSV) and os.path.exists(DRIVE_CKPT):
    os.makedirs(os.path.dirname(SCORES_CSV), exist_ok=True)
    shutil.copy2(DRIVE_CKPT, SCORES_CSV)
    print(f'Restored {SCORES_CSV} from Drive checkpoint ({DRIVE_CKPT}).')

# ----- Resume bookkeeping -------------------------------------------------
RESULT_COLS = ['model', 'image_id', 'fae_method', 'metric', 'score', 'error']
done_triples = set()
if os.path.exists(SCORES_CSV):
    _prev = pd.read_csv(SCORES_CSV)
    done_triples = set(map(tuple, _prev[['model', 'image_id', 'fae_method']].drop_duplicates().values))
    print(f'Resume: {len(done_triples)} (model, image, FAE) triples already done '
          f'(restored from Drive if this is a fresh runtime).')
else:
    print('No prior checkpoint (local or Drive) — starting fresh.')

def _append(rows):
    df = pd.DataFrame(rows, columns=RESULT_COLS)
    df.to_csv(SCORES_CSV, mode='a', header=not os.path.exists(SCORES_CSV), index=False)

# ----- Main loop ----------------------------------------------------------
t_start = time.time()
n_done = len(done_triples)
n_start = n_done   # triples already done before this session (for rate/ETA)
n_total = N_IMAGES * len(models_dict) * len(fae_names)

for img_idx in range(N_IMAGES):
    sample = dataset[img_idx]
    image_tensor = sample['image']
    image_id = sample['image_id']
    image_np = image_tensor.numpy()
    mask_np = sample['mask'].numpy() if sample['mask'] is not None else None
    t_img = time.time()

    for model_name, (model, arch) in models_dict.items():
        with torch.no_grad():
            target = int(model(image_tensor.unsqueeze(0).to(device)).argmax(dim=1).item())

        for fae_name in fae_names:
            if (model_name, image_id, fae_name) in done_triples:
                continue

            t_triple = time.time()
            rows = []
            scores = None
            try:
                # --- attribution (cached) ---
                if fae_name == 'gradcam':
                    tl = get_gradcam_target_layer(model, arch)
                    fae_hp = {'image_size': 224}
                    ckw = dict(model=model, image=image_tensor, target_layer=tl, device=device)
                    cfn = compute_gradcam
                elif fae_name == 'integrated_gradients':
                    fae_hp = {'n_steps': 50}
                    ckw = dict(model=model, image=image_tensor, device=device, n_steps=50)
                    cfn = compute_integrated_gradients
                elif fae_name == 'occlusion':
                    fae_hp = {'sliding_window_shapes': (3, 15, 15), 'strides': (3, 8, 8)}
                    ckw = dict(model=model, image=image_tensor, device=device,
                               sliding_window_shapes=(3, 15, 15), strides=(3, 8, 8))
                    cfn = compute_occlusion
                else:
                    fae_hp = {}
                    ckw = dict(model=model, image=image_tensor, device=device)
                    cfn = FAE_METHODS[fae_name]

                attr_np = compute_or_load(
                    cache=cache, compute_fn=cfn, model_arch=arch,
                    fae_method=fae_name, image_id=image_id, target=target,
                    fae_hyperparams=fae_hp, **ckw,
                ).numpy()

                explain_func = _make_explain_func(fae_name, model, arch, device)

                # --- metrics: capture failures PER METRIC ---
                scores, _t = compute_all_metrics(
                    model=model, image=image_np, attribution=attr_np, target=target,
                    mask=mask_np, device=device, explain_func=explain_func,
                    fae_method=fae_name, include_non_sensitivity=INCLUDE_NON_SENSITIVITY,
                    include_model_parameter_randomisation=INCLUDE_MPR,
                    return_timings=True,
                )
                for metric_name, score in scores.items():
                    # A NaN here is either an intentional skip (completeness on a
                    # non-completeness method, non_sensitivity disabled) or a real
                    # failure (MPR). compute_all_metrics already logged failures;
                    # record nan_reason so the coverage report can separate them.
                    err = ''
                    if isinstance(score, float) and np.isnan(score):
                        err = 'nan'
                    rows.append({'model': model_name, 'image_id': image_id,
                                 'fae_method': fae_name, 'metric': metric_name,
                                 'score': score, 'error': err})
            except Exception as exc:
                # Whole-FAE failure (e.g. attribution crashed): record one row per
                # metric so coverage accounting stays complete.
                msg = f'{type(exc).__name__}: {exc}'
                print(f'  FAIL {model_name}/{image_id}/{fae_name}: {msg}')
                traceback.print_exc()
                metric_names = list(scores.keys()) if isinstance(scores, dict) else ['attribution_failed']
                for metric_name in metric_names:
                    rows.append({'model': model_name, 'image_id': image_id,
                                 'fae_method': fae_name, 'metric': metric_name,
                                 'score': float('nan'), 'error': msg})

            _append(rows)
            done_triples.add((model_name, image_id, fae_name))
            n_done += 1
            dt_triple = time.time() - t_triple
            if dt_triple >= SLOW_TRIPLE_S:
                print(f'    {model_name}/{fae_name}: {dt_triple:.0f}s  [{image_id}]  {n_done}/{n_total}')

    # --- periodic Drive snapshot (durability) ---
    if (img_idx + 1) % SYNC_EVERY == 0:
        _sync_to_drive(SCORES_CSV, DRIVE_CKPT)

    new = n_done - n_start
    if new > 0 and (img_idx + 1) % LOG_EVERY == 0:
        el = time.time() - t_start
        rate = el / new                       # sec per newly-computed triple
        tpi = len(models_dict) * len(fae_names)
        eta_h = (n_total - n_done) * rate / 3600
        print(f'[img {img_idx+1:3d}/{N_IMAGES} {image_id}] '
              f'triples {n_done}/{n_total} ({100*n_done/n_total:.1f}%) | '
              f'img {time.time()-t_img:.0f}s | avg {rate*tpi/60:.1f} min/img | '
              f'ETA {eta_h:.1f}h | {datetime.datetime.now().strftime("%H:%M")}')

# --- final Drive snapshot (durability) ---
_sync_to_drive(SCORES_CSV, DRIVE_CKPT)

# --- protected final scores on Drive (what every downstream cell reads) ---
# Distinct from the rolling /checkpoints/ file. This is the ONLY write to
# SCORES_FINAL in the entire notebook; every other cell wraps its outputs in
# _safe_out() so this file can never be clobbered by a colliding output path.
try:
    os.makedirs(os.path.dirname(SCORES_FINAL), exist_ok=True)
    shutil.copy2(SCORES_CSV, SCORES_FINAL)
    print(f'Protected scores written to Drive: {SCORES_FINAL}')
except Exception as exc:
    print(f'  WARN: writing SCORES_FINAL failed (continuing): {type(exc).__name__}: {exc}')

print(f'\nDone. Total elapsed: {(time.time()-t_start)/60:.1f} min')
print(f'Scores CSV (local): {SCORES_CSV}')
print(f'Drive checkpoint:   {DRIVE_CKPT}')
print(f'Protected final:    {SCORES_FINAL}')

## 7. Coverage & NaN Report (incl. MPR)

Quantifies completeness of the scores CSV and isolates the
`model_parameter_randomisation` failure rate that plagued the pilot. Writes
`results/full_run_coverage_600.csv` (per model × metric).

In [ ]:
import pandas as pd, numpy as np, os

# Read scores from the durable, protected Drive file (with graceful fallback).
SCORES_CSV = _resolve_scores()
COVERAGE_CSV = _safe_out(f'{DRIVE_RESULTS}/full_run_coverage_600.csv')

df = pd.read_csv(SCORES_CSV)
print(f'Total score rows: {len(df)}')
print(f'Unique (model, image, FAE) triples: '
      f"{df[['model','image_id','fae_method']].drop_duplicates().shape[0]}")

# Real failures carry a non-empty, non-"nan" error string.
df['is_error'] = df['error'].fillna('').astype(str).str.len().gt(0) & \
                 (df['error'].fillna('') != 'nan')
df['is_nan'] = df['score'].isna()

cov = (df.groupby(['model', 'metric'])
         .agg(n=('score', 'size'),
              n_nan=('is_nan', 'sum'),
              n_error=('is_error', 'sum'))
         .reset_index())
cov['coverage'] = 1.0 - cov['n_nan'] / cov['n']
cov.to_csv(COVERAGE_CSV, index=False)

print('\n=== Coverage per (model, metric) — sorted by coverage ascending ===')
print(cov.sort_values('coverage').to_string(index=False))

print('\n=== model_parameter_randomisation coverage (the pilot problem) ===')
mpr = cov[cov['metric'] == 'model_parameter_randomisation']
print(mpr.to_string(index=False))

print('\n=== Distinct error messages (real failures only) ===')
errs = df[df['is_error']]['error'].value_counts().head(20)
print(errs.to_string() if not errs.empty else '  (no real failures recorded)')
print(f'\nCoverage CSV → {COVERAGE_CSV}')

## 8. Redundancy Matrices per Model

Reuses `src.metrics.redundancy.compute_redundancy_matrix` (Spearman, |ρ|>0.85,
variance pre-screen) to produce `redundancy_matrix_{resnet18,squeezenet}.csv`
— the same filenames Chapter 4 reads. Also prints the pruned pairs and the
intersection $M^*$.

In [ ]:
import pandas as pd
from src.metrics.redundancy import (
    compute_redundancy_matrix, prune_redundant_metrics, METRIC_CATEGORIES,
    category_redundancy_summary,
)

# Read scores from the durable, protected Drive file (with graceful fallback).
SCORES_CSV = _resolve_scores()
df = pd.read_csv(SCORES_CSV)[['model', 'image_id', 'fae_method', 'metric', 'score']]

mats = compute_redundancy_matrix(df, method='spearman', group_by=('model',))
kept_by_model = {}
for (model,), mat in mats.items():
    out = _safe_out(f'{DRIVE_RESULTS}/redundancy_matrix_{model}.csv')
    mat.to_csv(out)
    kept, pruned = prune_redundant_metrics(mat, threshold=0.85)
    kept_by_model[model] = set(kept)
    print(f'\n=== {model} ===  saved {out}')
    print('within-category mean |rho|:')
    print(category_redundancy_summary(mat, METRIC_CATEGORIES).to_string())
    print(f'kept (M* for {model}): {kept}')
    print(f'pruned pairs: {pruned}')

if kept_by_model:
    inter = set.intersection(*kept_by_model.values())
    print(f'\nIntersection M* (both models): {sorted(inter)}')

## 9. Meta-Evaluation — 600 Images (~4–5 h, resumable)

Noise Resilience (NR) + Adversarial Reactivity (AR) for every eligible
`(model, fae, metric)` triple (Hedström et al., TMLR 2024), via the same
`run_meta_evaluation_full` path as `colab_meta_evaluation.ipynb`. Checkpoints
after each `(model, FAE)` pair; re-run to resume.

In [ ]:
import sys, os, time, datetime, shutil
import numpy as np, torch, quantus, pandas as pd
sys.path.insert(0, '.')
os.makedirs('results', exist_ok=True)

# Read scores from the durable, protected Drive file (with graceful fallback).
SCORES_CSV   = _resolve_scores()
# Outputs are durable on Drive with DISTINCT basenames (never the scores file).
METAEVAL_OUT = _safe_out(f'{DRIVE_RESULTS}/meta_evaluation_reliability_FULL.csv')
PROGRESS_LOG = _safe_out(f'{DRIVE_RESULTS}/meta_eval_progress_FULL.log')

# ----- Drive-durable checkpoint -------------------------------------------
# run_meta_evaluation_full() already appends to METAEVAL_OUT after each
# (model, FAE) pair and resumes from any 'completed'/'skipped_nan' rows it finds
# there. METAEVAL_OUT now lives on Drive directly, so it is durable across a
# runtime recycle. We also keep a /checkpoints/ snapshot as belt-and-braces and
# restore from it if the durable output is somehow missing.
DRIVE_CKPT_OUT = f'{DRIVE_THESIS}/results/checkpoints/{os.path.basename(METAEVAL_OUT)}'
DRIVE_CKPT_LOG = f'{DRIVE_THESIS}/results/checkpoints/{os.path.basename(PROGRESS_LOG)}'

def _sync_to_drive(local, drive):
    """Push a full-file snapshot of *local* to *drive*; never crash the run."""
    try:
        os.makedirs(os.path.dirname(drive), exist_ok=True)
        shutil.copy2(local, drive)
        print(f'  synced {os.path.basename(local)} to Drive ({drive})')
    except Exception as exc:
        print(f'  WARN: Drive sync failed (continuing): {type(exc).__name__}: {exc}')

# ----- Restore-on-start: pull prior progress back from the checkpoint -------
for _local, _drive in [(METAEVAL_OUT, DRIVE_CKPT_OUT), (PROGRESS_LOG, DRIVE_CKPT_LOG)]:
    if not os.path.exists(_local) and os.path.exists(_drive):
        os.makedirs(os.path.dirname(_local), exist_ok=True)
        shutil.copy2(_drive, _local)
        print(f'Restored {_local} from Drive checkpoint ({_drive}).')

# Meta-eval screens on the *non-error* long table; drop the extra columns.
slice_df = pd.read_csv(SCORES_CSV)[['model', 'image_id', 'fae_method', 'metric', 'score']]
print(f'Loaded slice: {len(slice_df)} rows')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

from src.models.classifiers import load_resnet18, load_squeezenet
from src.data.isic_dataset import ISIC2017Dataset
from src.pipeline import _make_explain_func
from src.meta_evaluation.metaquantus_wrapper import (
    run_meta_evaluation_full, screen_meta_eval_candidates,
)

models = {
    'resnet18':   load_resnet18('weights/resnet18_isic2017.pth', device=device),
    'squeezenet': load_squeezenet('weights/squeezenet_isic2017.pth', device=device),
}

META_N_IMAGES = 64   # meta-validation runs on a representative SAMPLE, not all 600.
                     # NR/AR on 600 imgs ~ 90h; metric reliability is well estimated
                     # on a modest deterministic sample. Raise/lower as time allows.
dataset = ISIC2017Dataset(root_dir='data', split='test', image_size=224, return_mask=True)
_N_FULL = len(dataset)
_rng = np.random.default_rng(42)
_sel = sorted(_rng.choice(_N_FULL, size=min(META_N_IMAGES, _N_FULL), replace=False).tolist())
samples = [dataset[i] for i in _sel]
N_IMAGES = len(samples)
print(f'Meta-eval on {N_IMAGES}/{_N_FULL} sampled test images (seed 42).')
images = [s['image'] for s in samples]
masks  = [s['mask']  for s in samples]

_BATCH = 64
_img_t = torch.stack([s['image'] for s in samples]).to(device)
targets = []
with torch.no_grad():
    for _i in range(0, N_IMAGES, _BATCH):
        targets.extend(models['resnet18'](_img_t[_i:_i+_BATCH]).argmax(dim=1).tolist())
del _img_t
print(f'Targets computed for {len(targets)} images.')

_FAE_NAMES = ['integrated_gradients', 'saliency', 'gradcam', 'deep_lift',
              'guided_backprop', 'lrp', 'occlusion']
# Per-model explain funcs: Grad-CAM's target layer is architecture-specific,
# so each model needs funcs bound to ITS arch (resnet18-bound funcs raise
# 'SqueezeNet has no layer4'). run_meta_evaluation_full accepts this nested form.
fae_methods = {m: {n: _make_explain_func(n, models[m], m, device) for n in _FAE_NAMES}
               for m in models}

metric_fns = {
    'faithfulness_correlation': quantus.FaithfulnessCorrelation(
        nr_runs=100, subset_size=224, perturb_baseline='black',
        normalise=True, abs=False, return_aggregate=False, disable_warnings=True),
    'pixel_flipping': quantus.PixelFlipping(
        features_in_step=224, perturb_baseline='black', normalise=True, abs=False,
        return_aggregate=False, return_auc_per_sample=True, disable_warnings=True),
    'max_sensitivity': quantus.MaxSensitivity(
        nr_samples=10, lower_bound=0.2, normalise=False, abs=False,
        return_aggregate=False, disable_warnings=True),
    'avg_sensitivity': quantus.AvgSensitivity(
        nr_samples=10, lower_bound=0.2, normalise=False, abs=False,
        return_aggregate=False, disable_warnings=True),
    'relevance_mass_accuracy': quantus.RelevanceMassAccuracy(
        normalise=True, abs=False, return_aggregate=False, disable_warnings=True),
    'pointing_game': quantus.PointingGame(
        normalise=True, abs=True, return_aggregate=False, disable_warnings=True),
    'sparseness': quantus.Sparseness(
        abs=True, normalise=True, return_aggregate=False, disable_warnings=True),
    'complexity': quantus.Complexity(
        abs=True, normalise=True, return_aggregate=False, disable_warnings=True),
    'model_parameter_randomisation': quantus.ModelParameterRandomisation(
        layer_order='top_down', normalise=True, abs=True,
        return_average_correlation=True, return_aggregate=False, disable_warnings=True),
    'random_logit': quantus.RandomLogit(
        num_classes=3, abs=True, normalise=True,
        return_aggregate=False, disable_warnings=True),
    'completeness': quantus.Completeness(
        abs=False, normalise=False, perturb_baseline='black',
        return_aggregate=False, disable_warnings=True),
}

candidates = screen_meta_eval_candidates(slice_df, min_valid_fraction=0.5)
eligible = int(candidates['run_meta_eval'].sum())
print(f'Eligible triples: {eligible} | Skipped (low coverage): {len(candidates)-eligible}')

_done = 0
if os.path.exists(METAEVAL_OUT):
    _done = int(pd.read_csv(METAEVAL_OUT)['status'].isin(['completed','skipped_nan']).sum())
    print(f'Resume: {_done} triples already done (restored from Drive if fresh runtime).')
_remaining = max(0, eligible - _done)
print(f'Remaining: {_remaining} | est. ~{_remaining*50*50/3600:.1f} h on T4')
print('Safe to disconnect: checkpoints after each (model, FAE) pair.')
print(f'Starting at {datetime.datetime.now().isoformat(timespec="minutes")}\n')

try:
    result_df = run_meta_evaluation_full(
        vertical_slice_df=slice_df, models=models, metric_fns=metric_fns,
        fae_methods=fae_methods, images=images, targets=targets, masks=masks,
        device=device, n_seeds=5, n_levels=5,
        output_csv=METAEVAL_OUT, progress_log=PROGRESS_LOG,
    )
finally:
    # Sync whatever progress exists back to the /checkpoints/ snapshot — even if
    # interrupted mid-cell. (METAEVAL_OUT itself is already on Drive.)
    if os.path.exists(METAEVAL_OUT):
        _sync_to_drive(METAEVAL_OUT, DRIVE_CKPT_OUT)
    if os.path.exists(PROGRESS_LOG):
        _sync_to_drive(PROGRESS_LOG, DRIVE_CKPT_LOG)

print(f'\nThis session: {len(result_df)} triples processed.')
if os.path.exists(METAEVAL_OUT):
    print(pd.read_csv(METAEVAL_OUT)['status'].value_counts().to_string())

## 10. Summarise Meta-Evaluation

Per-metric NR / AR / combined reliability (completed triples). Safe to run at
any time during cell 9 to check partial progress.

In [ ]:
import pandas as pd, os
# Read the durable meta-eval output from Drive (fall back to local if absent).
METAEVAL_OUT = f'{DRIVE_RESULTS}/meta_evaluation_reliability_FULL.csv'
if not os.path.exists(METAEVAL_OUT):
    METAEVAL_OUT = 'results/meta_evaluation_reliability_FULL.csv'
print(f'Reading meta-eval from: {METAEVAL_OUT}')
rel = pd.read_csv(METAEVAL_OUT)
done = rel[rel['status'] == 'completed']
print(f'rows={len(rel)}  completed={len(done)}  '
      f"skipped_nan={(rel['status']=='skipped_nan').sum()}  "
      f"failed={(rel['status']=='failed').sum()}")
if len(done):
    summ = (done.groupby('metric')[['nr_score','ar_score','combined_reliability']]
                .mean().round(3).sort_values('combined_reliability', ascending=False))
    print('\n=== Mean reliability per metric ===')
    print(summ.to_string())

## 11. Four-Scheme Ranking Comparison

Runs `experiments/compare_rankings.py` on the FULL CSVs → uniform /
autoweighted / mqdiscount / single_fc effectiveness per (model, image, FAE).

In [ ]:
import os, pandas as pd

# Read scores from the durable, protected Drive file (with graceful fallback).
SCORES_CSV   = _resolve_scores()
# Meta-eval reliability lives durably on Drive (fall back to local if absent).
METAEVAL_OUT = f'{DRIVE_RESULTS}/meta_evaluation_reliability_FULL.csv'
if not os.path.exists(METAEVAL_OUT):
    METAEVAL_OUT = 'results/meta_evaluation_reliability_FULL.csv'
# Output is durable on Drive with a DISTINCT basename (never the scores file).
RANKING_OUT  = _safe_out(f'{DRIVE_RESULTS}/ranking_comparison_FULL.csv')

for _p in [SCORES_CSV, METAEVAL_OUT]:
    assert os.path.exists(_p), f'Missing input: {_p}'

# compare_rankings.py reads the 5-col long schema; strip the extra columns into
# a /tmp working file (temp stays local; source is the protected scores file).
_tmp = '/tmp/full_run_scores_long.csv'
pd.read_csv(SCORES_CSV)[['model','image_id','fae_method','metric','score']].to_csv(_tmp, index=False)

!python experiments/compare_rankings.py \
    --slice-csv       {_tmp} \
    --reliability-csv {METAEVAL_OUT} \
    --output-csv      {RANKING_OUT}

print('\nranking_comparison_FULL.csv head:')
print(pd.read_csv(RANKING_OUT).head().to_string(index=False))

## 12. NormEnsembleXAI Ensemble — score it (~1–1.5 h, resumable)

Computes the NormEnsembleXAI ensemble (Decision D6: Second-Moment normalise →
mean over all 7 FAE) per (model, image) via `src.attributions.ensembling`,
then scores it with the **same** 12 metrics. Output schema matches the main
scores CSV with `fae_method = 'ensemble'`, so it flows through the same ranking
and statistical-test code → enables the paired individual-vs-ensembled
comparison (Table tab:ensemble, tab:wilcoxon).

Checkpoints/resumes per (model, image). The ensemble explain_func recomputes
the full 7-method ensemble under perturbation, so robustness/randomisation
metrics are honest (not cheap).

In [ ]:
import os, sys, time, datetime, shutil
import numpy as np, pandas as pd, torch
sys.path.insert(0, '.')
os.makedirs('results', exist_ok=True)

ENSEMBLE_CSV = 'results/full_run_ensemble_7fae_12metrics_600.csv'  # fast local append
INCLUDE_NON_SENSITIVITY = False
INCLUDE_MPR = False               # see cell 6
SEED = 42

# ----- Drive-durable checkpoint + output ----------------------------------
# Fast local append + whole-file snapshot to Drive every SYNC_EVERY images, so a
# runtime recycle (which wipes /content) does not lose ensemble progress. The
# durable ensemble output (what cell 26 reads) lives on Drive with a DISTINCT
# basename, wrapped in _safe_out so it can never collide with the scores file.
ENSEMBLE_FINAL = _safe_out(f'{DRIVE_RESULTS}/full_run_ensemble_7fae_12metrics_600.csv')
DRIVE_CKPT = f'{DRIVE_THESIS}/results/checkpoints/{os.path.basename(ENSEMBLE_CSV)}'
SYNC_EVERY = 25   # snapshot local CSV -> Drive every N images (loss window <= N imgs)

def _sync_to_drive(local, drive):
    """Push a full-file snapshot of *local* to *drive*; never crash the run."""
    try:
        os.makedirs(os.path.dirname(drive), exist_ok=True)
        shutil.copy2(local, drive)
        n_rows = sum(1 for _ in open(local)) - 1  # minus header
        print(f'  synced {max(n_rows, 0)} rows to Drive ({drive})')
    except Exception as exc:
        print(f'  WARN: Drive sync failed (continuing): {type(exc).__name__}: {exc}')

from src.attributions.generate import (
    FAE_METHODS, compute_or_load, compute_gradcam, compute_integrated_gradients,
    compute_occlusion,
)
from src.attributions.cache import AttributionCache
from src.attributions.ensembling import ensemble_attributions_numpy
from src.data.isic_dataset import ISIC2017Dataset
from src.metrics.quantus_wrapper import compute_all_metrics
from src.models.classifiers import (
    get_gradcam_target_layer, load_resnet18, load_squeezenet,
)
from src.pipeline import _make_explain_func

import random
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
cache = AttributionCache(root='attributions_cache')

models_dict = {
    'resnet18':   (load_resnet18('weights/resnet18_isic2017.pth', device=device), 'resnet18'),
    'squeezenet': (load_squeezenet('weights/squeezenet_isic2017.pth', device=device), 'squeezenet'),
}
dataset = ISIC2017Dataset(root_dir='data', split='test', image_size=224, return_mask=True)
N_IMAGES = len(dataset)
fae_names = list(FAE_METHODS.keys())

# ----- Restore-on-start: pull prior progress back from Drive --------------
# Prefer the durable Drive output, then the rolling /checkpoints/ snapshot.
if not os.path.exists(ENSEMBLE_CSV):
    for _drive in (ENSEMBLE_FINAL, DRIVE_CKPT):
        if os.path.exists(_drive):
            os.makedirs(os.path.dirname(ENSEMBLE_CSV), exist_ok=True)
            shutil.copy2(_drive, ENSEMBLE_CSV)
            print(f'Restored {ENSEMBLE_CSV} from Drive ({_drive}).')
            break

RESULT_COLS = ['model', 'image_id', 'fae_method', 'metric', 'score', 'error']
done = set()
if os.path.exists(ENSEMBLE_CSV):
    _prev = pd.read_csv(ENSEMBLE_CSV)
    done = set(map(tuple, _prev[['model','image_id']].drop_duplicates().values))
    print(f'Resume: {len(done)} (model, image) ensembles done '
          f'(restored from Drive if this is a fresh runtime).')
else:
    print('No prior checkpoint (local or Drive) — starting fresh.')

def _attr_for(model, arch, image_tensor, image_id, fae_name, target):
    if fae_name == 'gradcam':
        tl = get_gradcam_target_layer(model, arch)
        return compute_or_load(cache=cache, compute_fn=compute_gradcam, model_arch=arch,
            fae_method=fae_name, image_id=image_id, target=target,
            fae_hyperparams={'image_size':224}, model=model, image=image_tensor,
            target_layer=tl, device=device).numpy()
    if fae_name == 'integrated_gradients':
        return compute_or_load(cache=cache, compute_fn=compute_integrated_gradients,
            model_arch=arch, fae_method=fae_name, image_id=image_id, target=target,
            fae_hyperparams={'n_steps':50}, model=model, image=image_tensor,
            device=device, n_steps=50).numpy()
    if fae_name == 'occlusion':
        return compute_or_load(cache=cache, compute_fn=compute_occlusion, model_arch=arch,
            fae_method=fae_name, image_id=image_id, target=target,
            fae_hyperparams={'sliding_window_shapes':(3,15,15),'strides':(3,8,8)},
            model=model, image=image_tensor, device=device,
            sliding_window_shapes=(3,15,15), strides=(3,8,8)).numpy()
    return compute_or_load(cache=cache, compute_fn=FAE_METHODS[fae_name], model_arch=arch,
        fae_method=fae_name, image_id=image_id, target=target, fae_hyperparams={},
        model=model, image=image_tensor, device=device).numpy()

def _make_ensemble_explain_func(model, arch):
    # Quantus explain_func: per perturbed input recompute all 7 FAE then ensemble.
    base = {n: _make_explain_func(n, model, arch, device) for n in fae_names}
    def _explain(model, inputs, targets, **kw):
        out = []
        for i in range(inputs.shape[0]):
            xi = inputs[i:i+1]
            ti = np.array([int(targets[i])])
            attr_dict = {n: base[n](model, xi, ti)[0] for n in fae_names}
            out.append(ensemble_attributions_numpy(attr_dict, methods=fae_names, aggregation='mean'))
        return np.stack(out, axis=0)
    return _explain

t0 = time.time()
for img_idx in range(N_IMAGES):
    sample = dataset[img_idx]
    image_tensor = sample['image']; image_id = sample['image_id']
    image_np = image_tensor.numpy()
    mask_np = sample['mask'].numpy() if sample['mask'] is not None else None

    for model_name, (model, arch) in models_dict.items():
        if (model_name, image_id) in done:
            continue
        with torch.no_grad():
            target = int(model(image_tensor.unsqueeze(0).to(device)).argmax(dim=1).item())

        rows = []
        try:
            attr_dict = {n: _attr_for(model, arch, image_tensor, image_id, n, target)
                         for n in fae_names}
            ens_attr = ensemble_attributions_numpy(attr_dict, methods=fae_names, aggregation='mean')
            ens_explain = _make_ensemble_explain_func(model, arch)
            scores = compute_all_metrics(
                model=model, image=image_np, attribution=ens_attr, target=target,
                mask=mask_np, device=device, explain_func=ens_explain,
                fae_method='ensemble', include_non_sensitivity=INCLUDE_NON_SENSITIVITY,
                                include_model_parameter_randomisation=INCLUDE_MPR)
            for metric_name, score in scores.items():
                err = 'nan' if (isinstance(score, float) and np.isnan(score)) else ''
                rows.append({'model': model_name, 'image_id': image_id,
                             'fae_method': 'ensemble', 'metric': metric_name,
                             'score': score, 'error': err})
        except Exception as exc:
            msg = f'{type(exc).__name__}: {exc}'
            print(f'  FAIL ensemble {model_name}/{image_id}: {msg}')
            rows.append({'model': model_name, 'image_id': image_id,
                         'fae_method': 'ensemble', 'metric': 'ensemble_failed',
                         'score': float('nan'), 'error': msg})
        pd.DataFrame(rows, columns=RESULT_COLS).to_csv(
            ENSEMBLE_CSV, mode='a', header=not os.path.exists(ENSEMBLE_CSV), index=False)
        done.add((model_name, image_id))

    # --- periodic Drive snapshot (durability): rolling checkpoint + durable out ---
    if (img_idx + 1) % SYNC_EVERY == 0:
        _sync_to_drive(ENSEMBLE_CSV, DRIVE_CKPT)
        _sync_to_drive(ENSEMBLE_CSV, _safe_out(ENSEMBLE_FINAL))

    if (img_idx + 1) % 10 == 0 or img_idx == 0:
        print(f'[{img_idx+1:3d}/{N_IMAGES}] ensembles done={len(done)} '
              f'elapsed={(time.time()-t0)/60:.1f} min '
              f'@ {datetime.datetime.now().isoformat(timespec="minutes")}')

# --- final Drive snapshot (durability): rolling checkpoint + durable output ---
if os.path.exists(ENSEMBLE_CSV):
    _sync_to_drive(ENSEMBLE_CSV, DRIVE_CKPT)
    _sync_to_drive(ENSEMBLE_CSV, _safe_out(ENSEMBLE_FINAL))
print(f'\nEnsemble scoring done: {(time.time()-t0)/60:.1f} min → {ENSEMBLE_CSV}')
print(f'Drive checkpoint: {DRIVE_CKPT}')
print(f'Durable output:   {ENSEMBLE_FINAL}')

## 13. Ensemble Four-Scheme Ranking

Runs the same four-scheme aggregation over a long table that contains **both**
the 7 individual methods and the `ensemble` pseudo-method, then writes
`ensemble_ranking_FULL.csv` (rows for `fae_method == 'ensemble'`). This is the
input the local Wilcoxon test pairs against the best individual method.

In [ ]:
import os, pandas as pd

# Read scores from the durable, protected Drive file (with graceful fallback).
SCORES_CSV   = _resolve_scores()
# Durable ensemble scores from Drive (fall back to local if absent).
ENSEMBLE_CSV = f'{DRIVE_RESULTS}/full_run_ensemble_7fae_12metrics_600.csv'
if not os.path.exists(ENSEMBLE_CSV):
    ENSEMBLE_CSV = 'results/full_run_ensemble_7fae_12metrics_600.csv'
print(f'Reading ensemble scores from: {ENSEMBLE_CSV}')
# Meta-eval reliability lives durably on Drive (fall back to local if absent).
METAEVAL_OUT = f'{DRIVE_RESULTS}/meta_evaluation_reliability_FULL.csv'
if not os.path.exists(METAEVAL_OUT):
    METAEVAL_OUT = 'results/meta_evaluation_reliability_FULL.csv'
# Output is durable on Drive with a DISTINCT basename (never the scores file).
ENSEMBLE_RANK_OUT = _safe_out(f'{DRIVE_RESULTS}/ensemble_ranking_FULL.csv')

cols = ['model','image_id','fae_method','metric','score']
ind = pd.read_csv(SCORES_CSV)[cols]
ens = pd.read_csv(ENSEMBLE_CSV)[cols]
combined = pd.concat([ind, ens], ignore_index=True)
_tmp = '/tmp/combined_with_ensemble_long.csv'
combined.to_csv(_tmp, index=False)

_full_rank = '/tmp/ranking_with_ensemble.csv'
!python experiments/compare_rankings.py \
    --slice-csv       {_tmp} \
    --reliability-csv {METAEVAL_OUT} \
    --output-csv      {_full_rank}

rank = pd.read_csv(_full_rank)
rank[rank['fae_method'] == 'ensemble'].to_csv(ENSEMBLE_RANK_OUT, index=False)
print(f'Saved ensemble rows → {ENSEMBLE_RANK_OUT}')
print(rank[rank['fae_method']=='ensemble'].head().to_string(index=False))

## 14. Wilcoxon Sanity Check (mqdiscount vs single_fc)

Quick on-Colab sanity test using the new `wilcoxon_paired` helper. The full
Friedman/Nemenyi/Wilcoxon analysis for the thesis is run **locally** afterwards
via `experiments/run_statistical_analysis.py` (see the final checklist).

In [ ]:
import sys, os, pandas as pd
sys.path.insert(0, '.')
from src.comparison.statistical_tests import wilcoxon_paired

# Durable ranking output from Drive (fall back to local if absent).
RANKING_OUT = f'{DRIVE_RESULTS}/ranking_comparison_FULL.csv'
if not os.path.exists(RANKING_OUT):
    RANKING_OUT = 'results/ranking_comparison_FULL.csv'
print(f'Reading ranking from: {RANKING_OUT}')
df = pd.read_csv(RANKING_OUT)
res = wilcoxon_paired(df['effectiveness_mqdiscount'], df['effectiveness_single_fc'],
                      alternative='two-sided')
print('Wilcoxon mqdiscount vs single_fc:')
for k, v in res.items():
    print(f'  {k}: {v}')

## 15. Copy All Results to Drive

Saves every full-run artifact to `MyDrive/thesis/results/` for local download.
Also persists the test-split images/masks if not already on Drive.

In [ ]:
import shutil, os

# All downstream artifacts are now written DIRECTLY to Drive (DRIVE_RESULTS) by
# their producing cells, so this is a verify/summary step — not a bulk copy.
# Each entry maps a durable Drive path -> a local fallback to copy from if the
# Drive file is somehow missing. Every copy target is wrapped in _safe_out so it
# can never clobber the protected scores file.
_durable = [
    (SCORES_FINAL,                                          SCORES_LOCAL,                                          'scores (PROTECTED)'),
    (f'{DRIVE_RESULTS}/full_run_coverage_600.csv',          'results/full_run_coverage_600.csv',                  'coverage'),
    (f'{DRIVE_RESULTS}/redundancy_matrix_resnet18.csv',     'results/redundancy_matrix_resnet18.csv',             'redundancy resnet18'),
    (f'{DRIVE_RESULTS}/redundancy_matrix_squeezenet.csv',   'results/redundancy_matrix_squeezenet.csv',           'redundancy squeezenet'),
    (f'{DRIVE_RESULTS}/meta_evaluation_reliability_FULL.csv','results/meta_evaluation_reliability_FULL.csv',       'meta-eval'),
    (f'{DRIVE_RESULTS}/meta_eval_progress_FULL.log',        'results/meta_eval_progress_FULL.log',                'meta-eval log'),
    (f'{DRIVE_RESULTS}/ranking_comparison_FULL.csv',        'results/ranking_comparison_FULL.csv',                'ranking'),
    (f'{DRIVE_RESULTS}/full_run_ensemble_7fae_12metrics_600.csv', 'results/full_run_ensemble_7fae_12metrics_600.csv', 'ensemble scores'),
    (f'{DRIVE_RESULTS}/ensemble_ranking_FULL.csv',          'results/ensemble_ranking_FULL.csv',                  'ensemble ranking'),
]

print('=== Durable artifact verification (Drive) ===')
for drive_path, local_fallback, label in _durable:
    if os.path.exists(drive_path):
        print(f'OK   {label}: {drive_path} ({os.path.getsize(drive_path)//1024} KB)')
    elif os.path.exists(local_fallback):
        # Still-local artifact: copy it up to Drive. _safe_out refuses any path
        # that resolves to SCORES_FINAL EXCEPT — for the scores row only — the
        # producing cell already wrote SCORES_FINAL; here we never overwrite it.
        if os.path.abspath(drive_path) == os.path.abspath(SCORES_FINAL):
            # Scores: only copy up if SCORES_FINAL truly does not exist yet.
            os.makedirs(os.path.dirname(drive_path), exist_ok=True)
            shutil.copy2(local_fallback, drive_path)
            print(f'COPIED scores from local -> {drive_path} (SCORES_FINAL was missing)')
        else:
            dest = _safe_out(drive_path)
            os.makedirs(os.path.dirname(dest), exist_ok=True)
            shutil.copy2(local_fallback, dest)
            print(f'COPIED {label}: local -> {dest} ({os.path.getsize(dest)//1024} KB)')
    else:
        print(f'MISSING {label}: neither {drive_path} nor {local_fallback}')

# Persist the test-split images/masks if not already on Drive (unchanged).
for _d in ['data/images/test', 'data/masks/test']:
    _dest = f'{DRIVE_THESIS}/{_d}'
    if os.path.exists(_d) and not os.path.exists(_dest):
        os.makedirs(_dest, exist_ok=True)
        !cp -r {_d}/* {_dest}/
        print(f'Saved {_d} → Drive')

print('\nAll artifacts durable on Drive. Download results/ locally, then run:')
print('  .venv/bin/python experiments/run_statistical_analysis.py \\')
print('      --ranking-csv ../results/ranking_comparison_FULL.csv \\')
print('      --ensemble-csv ../results/ensemble_ranking_FULL.csv')
print('  .venv/bin/python experiments/render_results_tables.py \\')
print('      --slice-csv ../results/full_run_7fae_12metrics_600.csv \\')
print('      --reliability-csv ../results/meta_evaluation_reliability_FULL.csv \\')
print('      --ranking-csv ../results/ranking_comparison_FULL.csv \\')
print('      --ensemble-ranking-csv ../results/ensemble_ranking_FULL.csv \\')
print('      --stats-csv ../results/statistical_tests.csv \\')
print('      --n-images-label "full run, 600 images"')